In [ ]:
pip install imbalanced-learn

# 🧠 Subfield Classifier – Computer Science (CS)

This notebook sets up subfield classification for **CS abstracts** using the full 105-abstract dataset. 

### 💻 Step 1: Load and Filter CS Abstracts

We begin by loading the full dataset containing subfield labels and filtering it to include only the **Computer Science (CS)** abstracts. This sets the stage for subfield-level classification using a supervised learning pipeline.

In [ ]:
import pandas as pd

# Load the full dataset with subfield labels
df = pd.read_csv("data/NLP_Abstract_Dataset (Subfield)(105).csv")

# Filter only Computer Science records
cs_df = df[df["Discipline"] == "CS"].copy()

# Preview first few CS records
cs_df.head()

### 🧠 Step 2: Preprocessing – TF-IDF + Subfield Label Encoding

We transform the abstract text into numerical features using **TF-IDF vectorization**, and encode the subfield labels into integers. These are necessary preprocessing steps for training most machine learning models.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Extract abstract and subfield labels
X = cs_df['Abstract']
y = cs_df['Subfield']

# Use bigrams with tuned parameters
vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english', max_df=0.95, min_df=2)
X_tfidf = vectorizer.fit_transform(X)

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Check shapes
print("TF-IDF shape:", X_tfidf.shape)
print("Classes:", list(label_encoder.classes_))

### 🤖 Step 3: Train-Test Split and Model Training (SVM + SMOTE)

We split the dataset using a stratified 80/20 train-test split to preserve the distribution of subfields across the CS abstracts.

To address the significant class imbalance, we apply SMOTE (Synthetic Minority Oversampling Technique) with `k_neighbors=1` on the training set. This synthetically generates additional examples of underrepresented subfields before model training.

We then train a Support Vector Machine (SVM) classifier (`LinearSVC`) using bigram-enhanced TF-IDF features. This configuration is designed to maximize boundary separation between subfields while preserving class balance and semantic richness.

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.svm import LinearSVC

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Apply SMOTE to training set only
smote = SMOTE(random_state=42, k_neighbors=1)
X_train, y_train = smote.fit_resample(X_train.toarray(), y_train)

print("After SMOTE - Training shape:", X_train.shape)

# Train the SVM model on SMOTE-resampled data
svm_model = LinearSVC(max_iter=1000, random_state=42)
svm_model.fit(X_train, y_train)

# Predict on original test set
y_pred = svm_model.predict(X_test)

### 📊 Step 4: Model Evaluation – Classification Report & Confusion Matrix

We evaluate the model using precision, recall, F1-score, and a confusion matrix. These metrics help us understand which subfields the model is predicting correctly and where it's struggling.

In [ ]:
# 📊 Step 4: Model Evaluation — Classification Report & Confusion Matrix

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Print classification metrics
print("Classification Report:\n", classification_report(
    y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Create confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted Subfield')
plt.ylabel('True Subfield')
plt.title('Confusion Matrix – CS Subfield Classifier (SVM + SMOTE)')
plt.tight_layout()
plt.show()

📊 ***Confusion Matrix Interpretation*** – CS Subfield Classifier (SVM + SMOTE)

This confusion matrix shows the performance of the SVM-based CS subfield classifier after applying SMOTE to balance class distribution across 35 CS abstracts.

- **CV (Computer Vision)** was predicted perfectly.
- **CYB (Cybersecurity)** was predicted with perfect precision, though one true CYB instance was misclassified.
- **AI and CYB** had strong F1-scores with at least 50% recall.
- **ML and PAST** were not correctly classified, likely due to:
  - Very low number of training examples (even after SMOTE)
  - Sparse or overlapping language patterns in those abstracts
  - Insufficient generalization from synthetic examples
  
This result demonstrates that SMOTE did not reduce performance, and helped maintain or slightly improve macro-averaged metrics. Future improvements would require:
- Expanding the dataset for ML and PAST
- Using title/keyword features
- Testing advanced models like ensemble SVM or transformers 

Despite limitations, the pipeline is now structurally complete and correctly implemented for this phase.

### 🧾 Step 5: Save Final Model and TF-IDF Vectorizer

We save both the trained Logistic Regression model and the TF-IDF vectorizer to disk using `joblib`. This allows for consistent inference or retraining later without needing to repeat the full pipeline.

In [ ]:
import joblib

joblib.dump(svm_model, "cs_subfield_classifier_svm_smote.pkl")
joblib.dump(vectorizer, "cs_subfield_vectorizer_smote.pkl")

## ✅ Final Notes

This notebook completes the subfield classification pipeline for CS abstracts using a Support Vector Machine (SVM) classifier combined with bigram-enhanced TF-IDF features and SMOTE-based class balancing.

Although performance remains limited by dataset size, the final model achieved 50% accuracy and strong F1-scores for CV, CYB, and AI. SMOTE helped maintain performance stability but could not fully recover classes with very few training examples (ML, PAST).

Despite these constraints, this notebook confirms the successful and complete implementation of the subfield classification pipeline for the CS discipline. The same structure will now be extended to IS and IT in the next phases.